# 06 · Métricas de la industria: AUC, Gini, KS y estabilidad

**Objetivo:** medir, en la **muestra de prueba** (clientes que ningún modelo vio), qué tan bien separan el scorecard y LightGBM a los clientes buenos de los malos, y si el scorecard es estable.

## Conceptos

### AUC (área bajo la curva ROC)
Si se toma al azar un cliente malo y uno bueno, el AUC es la **probabilidad de que el modelo le asigne más riesgo al malo**.
- 0.5 = igual que lanzar una moneda.
- 1.0 = separación perfecta.
- Como referencia general (Hosmer y Lemeshow): 0.7–0.8 aceptable, 0.8–0.9 excelente, más de 0.9 sobresaliente.

### Gini
Es el AUC reescalado: **Gini = 2 × AUC − 1**. Va de 0 (al azar) a 1 (perfecto). Es la métrica más usada en bancos para reportar el poder de un scorecard.

### KS (Kolmogorov-Smirnov)
Se ordenan los clientes por puntaje y se acumulan, por separado, el % de malos y el % de buenos. El KS es la **mayor distancia entre esas dos curvas**: el punto del puntaje donde el modelo mejor separa a los malos de los buenos. Se usa mucho para elegir puntos de corte.

### PSI (*Population Stability Index*, índice de estabilidad)
Compara cómo se distribuyen los clientes entre dos muestras (por ejemplo, entrenamiento contra prueba, o el mes de desarrollo contra el mes actual). Detecta si la población cambió y el modelo podría dejar de funcionar.

| PSI | Interpretación |
|---|---|
| Menos de 0.10 | Estable |
| 0.10 a 0.25 | Revisar |
| Más de 0.25 | Inestable: el modelo probablemente necesita recalibrarse |

## Entorno

In [ ]:
import os
import sys
from pathlib import Path

USUARIO_GITHUB = "emilianomtz0189"
NOMBRE_REPO = "credit-risk-scorecard"
CARPETA_DRIVE = Path("/content/drive/MyDrive/credit-risk-scorecard")  # Donde está cs-training.csv

EN_COLAB = "google.colab" in sys.modules

if EN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

    ruta_repo = Path("/content") / NOMBRE_REPO
    if not ruta_repo.exists():
        !git clone -q https://github.com/{USUARIO_GITHUB}/{NOMBRE_REPO}.git {ruta_repo}
    else:
        !git -C {ruta_repo} pull -q
    os.chdir(ruta_repo)
elif Path.cwd().name == "notebooks":
    os.chdir("..")

sys.path.insert(0, str(Path.cwd()))
print("Directorio de trabajo:", Path.cwd())

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from src import config
from src.comparacion import construir_modelo_comparacion
from src.datos import copiar_datos_crudos
from src.limpieza import obtener_datos_limpios
from src.metricas import (
    bootstrap_diferencia_auc,
    estabilidad,
    graficar_ks,
    graficar_roc,
    psi_puntaje,
    psi_variables,
    tabla_deciles,
    tabla_metricas,
)
from src.scorecard import construir_scorecard

config.fijar_semilla()
config.crear_carpetas()
pd.set_option("display.float_format", "{:,.4f}".format)

In [ ]:
# Reconstruye ambos modelos de las etapas 4 y 5 (deterministas). Tarda unos minutos.
copiar_datos_crudos(CARPETA_DRIVE)
train, test = obtener_datos_limpios()
Y = config.COLUMNA_OBJETIVO

scorecard = construir_scorecard()
lightgbm = construir_modelo_comparacion()

puntaje_train = scorecard["puntaje_train"]["puntaje"]
puntaje_test = scorecard["puntaje_test"]["puntaje"]
print("Modelos listos.")

## 1. AUC, Gini y KS de ambos modelos

Para el scorecard se usa el puntaje con signo negativo, porque en las métricas "mayor valor = mayor riesgo". Se calculan en entrenamiento y en prueba: si un modelo es mucho mejor en entrenamiento que en prueba, está **sobreajustado** (memorizó en lugar de aprender).

In [ ]:
resultados = tabla_metricas({
    ("Scorecard", "entrenamiento"): (train[Y], -puntaje_train),
    ("Scorecard", "prueba"): (test[Y], -puntaje_test),
    ("LightGBM", "entrenamiento"): (train[Y], lightgbm["pd_train"]),
    ("LightGBM", "prueba"): (test[Y], lightgbm["pd_test"]),
})
resultados

In [ ]:
# Diferencia entre entrenamiento y prueba (sobreajuste)
brecha = (resultados.xs("entrenamiento", level="muestra") - resultados.xs("prueba", level="muestra"))
brecha.columns = [f"{c}_entrenamiento_menos_prueba" for c in brecha.columns]
brecha

### ¿La diferencia entre modelos es real o casualidad?

Se remuestrean los clientes de prueba 300 veces (*bootstrap*) y se calcula la diferencia de AUC en cada repetición. Si el intervalo de 95 % no incluye el cero, la diferencia es estadísticamente significativa. Después hay que preguntarse si también es **relevante para el negocio**.

In [ ]:
diferencia = bootstrap_diferencia_auc(test[Y], -puntaje_test, lightgbm["pd_test"])
print("AUC LightGBM − AUC scorecard, en prueba:")
diferencia

In [ ]:
graficar_roc({
    "Scorecard": (test[Y], -puntaje_test),
    "LightGBM": (test[Y], lightgbm["pd_test"]),
})
plt.show()

**Cómo leer la curva ROC:** cada punto es un posible punto de corte. El eje horizontal es el % de clientes buenos que se rechazarían y el vertical, el % de malos que se detectarían. Mientras más se acerque la curva a la esquina superior izquierda, mejor. La diagonal punteada es un modelo al azar.

## 2. KS del scorecard

In [ ]:
graficar_ks(test[Y], puntaje_test)
plt.show()

## 3. Tabla de deciles (tabla de ganancias)

Se ordena a los clientes de prueba del más riesgoso al menos riesgoso y se divide en 10 grupos iguales. La columna clave es `pct_malos_acum`: **qué porcentaje de todos los malos se concentra en los clientes más riesgosos**. Es la tabla que se presenta a la dirección para decidir a cuántos clientes rechazar o revisar.

In [ ]:
print("Scorecard (prueba)")
deciles_scorecard = tabla_deciles(test[Y], -puntaje_test)
display(deciles_scorecard)

print("LightGBM (prueba)")
deciles_lgb = tabla_deciles(test[Y], lightgbm["pd_test"])
display(deciles_lgb)

In [ ]:
captura = pd.DataFrame({
    "scorecard": deciles_scorecard["pct_malos_acum"],
    "lightgbm": deciles_lgb["pct_malos_acum"],
})
captura.index.name = "% de clientes más riesgosos"
captura.index = [f"{d * 10} %" for d in captura.index]
print("% de todos los malos capturados al revisar al X % de clientes más riesgosos:")
captura

## 4. Estabilidad del scorecard (PSI)

Se compara la distribución de **entrenamiento** (la "población de desarrollo") con la de **prueba**. Como ambas vienen de la misma base y se separaron al azar, se espera estabilidad; el valor de este análisis es dejar el procedimiento listo para monitorear el modelo cada mes cuando esté en producción.

In [ ]:
psi_total, distribucion = psi_puntaje(puntaje_train, puntaje_test)
print(f"PSI del puntaje: {psi_total:.4f} ({estabilidad(psi_total)})")
distribucion

In [ ]:
# PSI por variable (también llamado CSI): estabilidad de cada característica del scorecard
psi_variables(scorecard["agrupador"], train, test, scorecard["variables"])

## Resumen de la etapa

- Se midieron AUC, Gini y KS del scorecard y de LightGBM en entrenamiento y prueba.
- Se probó si la diferencia entre modelos es estadísticamente significativa (bootstrap).
- Se construyó la tabla de deciles: qué porcentaje de malos se concentra en los clientes más riesgosos.
- Se revisó la estabilidad del puntaje y de cada variable con el PSI.

**Siguiente paso:** notebook `07_perdida_esperada.ipynb`, pérdida esperada (PD × LGD × EAD) y propuesta de puntos de corte.